In [1]:
import sys
print(sys.executable)

/usr/local/opt/python@3.11/bin/python3.11


In [2]:
# %pip install pyspark==3.5.6

In [3]:
import pyspark
print(pyspark.__version__)
print(pyspark.__file__)

3.5.6
/usr/local/lib/python3.11/site-packages/pyspark/__init__.py


In [4]:
from pyspark.sql import SparkSession
spark = (
    SparkSession.builder
    .master("local[*]")
    .appName("Test")
    .getOrCreate()
)

26/09/06 00:34:11 WARN Utils: Your hostname, Ryans-MacBook-Air.local resolves to a loopback address: 127.0.0.1; using 192.168.1.82 instead (on interface en0)
26/09/06 00:34:11 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/09/06 00:34:12 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [28]:
from functools import reduce
from pathlib import Path
from pyspark.sql import DataFrame
from pyspark.sql.functions import (
    add_months,
    avg,
    col,
    concat,
    count,
    countDistinct,
    current_date,
    date_format,
    floor,
    lit,
    lpad,
    make_date,
    max,
    min,
    months_between,
    percentile_approx,
    regexp_replace,
    row_number,
    sha2,
    stddev,
    substring,
    to_date,
    when,
)
from pyspark.sql.types import (
    DateType,
    DoubleType,
    FloatType,
    IntegerType,
    LongType,
    StringType,
    StructField,
    StructType,
    TimestampType,
)
from pyspark.sql.window import Window


# HDB Resale Flat Data Pipeline

This notebook builds a PySpark data-cleaning and transformation pipeline for HDB resale flat transactions.

## Objectives
- Load and standardise multiple CSV files into one master dataset
- Compare and reconcile schemas across source files
- Profile data quality and detect anomalies
- Validate records using business rules
- Compute remaining lease duration
- Generate a stable resale identifier and hashed version

## Data source
- Folder: `Raw/`
- Format: CSV files with headers
- Tooling: PySpark 3.x

## Expected workflow
1. Read all source files and compare schemas
2. Align columns across files
3. Merge into a single dataset
4. Profile missing/null/duplicate values
5. Validate records against rules
6. Compute remaining lease and deduplicate records
7. Create transformed output with a resale identifier

## Notes
- All source files are assumed to share the same business domain but may differ slightly in schema.
- The final output is intended for further analysis or downstream reporting.


# Data Extraction

## 1. Combine Datasets to a Single Master Dataset

This section combines all monthly CSV files into a single master dataset.

### What this step does
- Reads every file in `Raw/`
- Captures each file's schema
- Identifies missing columns, extra columns, and type mismatches
- Normalises the schema before unioning datasets

### Why it matters
Different files may not always have identical column names or field types. This step ensures the combined dataset is structurally consistent before downstream analysis.


In [6]:
def compare_schemas(folder_path: str):
    folder = Path(folder_path)
    schemas = {}
    for file in sorted(folder.glob("*.csv")):
        df = (
            spark.read
            .option("header", True)
            .option("inferSchema", True)
            .csv(str(file))
        )
        schemas[file.name] = {
            field.name: field.dataType.simpleString()
            for field in df.schema.fields
        }
    return schemas
schemas = compare_schemas("Raw")
files = list(schemas.keys())
reference = files[0]
print(f"Reference file: {reference}")
print()
for file in files[1:]:
    ref_schema = schemas[reference]
    current_schema = schemas[file]
    missing = set(ref_schema) - set(current_schema)
    extra = set(current_schema) - set(ref_schema)
    type_differences = {
        column: (ref_schema[column], current_schema[column])
        for column in ref_schema.keys() & current_schema.keys()
        if ref_schema[column] != current_schema[column]
    }
    if not missing and not extra and not type_differences:
        print(f"✅ {file}: same schema")
    else:
        print(f"\n❌ {file}: different schema")
        if missing:
            print("   Missing columns:", sorted(missing))
        if extra:
            print("   Extra columns:", sorted(extra))
        if type_differences:
            print("   Data type differences:")
            for column, types in type_differences.items():
                print(f"      {column}: {types[0]} → {types[1]}")


Reference file: Resale Flat Prices (Based on Approval Date), 1990 - 1999.csv


❌ Resale Flat Prices (Based on Approval Date), 2000 - Feb 2012.csv: different schema
   Data type differences:
      resale_price: int → double

❌ Resale Flat Prices (Based on Registration Date), From Jan 2015 to Dec 2016.csv: different schema
   Extra columns: ['remaining_lease']
   Data type differences:
      resale_price: int → double

❌ Resale Flat Prices (Based on Registration Date), From Mar 2012 to Dec 2014.csv: different schema
   Data type differences:
      resale_price: int → double

❌ Resale flat prices based on registration date from Jan-2017 onwards.csv: different schema
   Extra columns: ['remaining_lease']
   Data type differences:
      resale_price: int → double


In [7]:
def infer_combined_schema(folder_path: str):
    folder = Path(folder_path)
    type_rank = {
        IntegerType: 1,
        LongType: 2,
        FloatType: 3,
        DoubleType: 4,
        DateType: 5,
        TimestampType: 6,
        StringType: 7,
    }
    schema_dict = {}
    for file in sorted(folder.glob("*.csv")):
        print("\n" + "=" * 80)
        print(f"FILE: {file.name}")
        print("=" * 80)
        df = (
            spark.read
            .option("header", True)
            .option("inferSchema", True)
            .csv(str(file))
        )
        print("Columns:")
        print(df.columns)
        print("\nSchema:")
        df.printSchema()
        print(f"Rows: {df.count()}")
        for field in df.schema.fields:
            column_name = field.name
            data_type = type(field.dataType)

            if column_name not in schema_dict:
                schema_dict[column_name] = field.dataType
            else:
                existing_type = type(schema_dict[column_name])
                if type_rank[data_type] > type_rank[existing_type]:
                    schema_dict[column_name] = field.dataType
    schema = StructType([
        StructField(column_name, data_type, True)
        for column_name, data_type in schema_dict.items()
    ])
    print("\n" + "=" * 80)
    print("FINAL SCHEMA")
    print("=" * 80)
    print(schema)
    return schema
schema = infer_combined_schema("Raw")



FILE: Resale Flat Prices (Based on Approval Date), 1990 - 1999.csv


Columns:
['month', 'town', 'flat_type', 'block', 'street_name', 'storey_range', 'floor_area_sqm', 'flat_model', 'lease_commence_date', 'resale_price']

Schema:
root
 |-- month: timestamp (nullable = true)
 |-- town: string (nullable = true)
 |-- flat_type: string (nullable = true)
 |-- block: string (nullable = true)
 |-- street_name: string (nullable = true)
 |-- storey_range: string (nullable = true)
 |-- floor_area_sqm: double (nullable = true)
 |-- flat_model: string (nullable = true)
 |-- lease_commence_date: integer (nullable = true)
 |-- resale_price: integer (nullable = true)



Rows: 287196

FILE: Resale Flat Prices (Based on Approval Date), 2000 - Feb 2012.csv


Columns:
['month', 'town', 'flat_type', 'block', 'street_name', 'storey_range', 'floor_area_sqm', 'flat_model', 'lease_commence_date', 'resale_price']

Schema:
root
 |-- month: timestamp (nullable = true)
 |-- town: string (nullable = true)
 |-- flat_type: string (nullable = true)
 |-- block: string (nullable = true)
 |-- street_name: string (nullable = true)
 |-- storey_range: string (nullable = true)
 |-- floor_area_sqm: double (nullable = true)
 |-- flat_model: string (nullable = true)
 |-- lease_commence_date: integer (nullable = true)
 |-- resale_price: double (nullable = true)

Rows: 369651

FILE: Resale Flat Prices (Based on Registration Date), From Jan 2015 to Dec 2016.csv
Columns:
['month', 'town', 'flat_type', 'block', 'street_name', 'storey_range', 'floor_area_sqm', 'flat_model', 'lease_commence_date', 'remaining_lease', 'resale_price']

Schema:
root
 |-- month: timestamp (nullable = true)
 |-- town: string (nullable = true)
 |-- flat_type: string (nullable = true)
 |-- bloc

Columns:
['month', 'town', 'flat_type', 'block', 'street_name', 'storey_range', 'floor_area_sqm', 'flat_model', 'lease_commence_date', 'remaining_lease', 'resale_price']

Schema:
root
 |-- month: timestamp (nullable = true)
 |-- town: string (nullable = true)
 |-- flat_type: string (nullable = true)
 |-- block: string (nullable = true)
 |-- street_name: string (nullable = true)
 |-- storey_range: string (nullable = true)
 |-- floor_area_sqm: double (nullable = true)
 |-- flat_model: string (nullable = true)
 |-- lease_commence_date: integer (nullable = true)
 |-- remaining_lease: string (nullable = true)
 |-- resale_price: double (nullable = true)

Rows: 239467

FINAL SCHEMA
StructType([StructField('month', TimestampType(), True), StructField('town', StringType(), True), StructField('flat_type', StringType(), True), StructField('block', StringType(), True), StructField('street_name', StringType(), True), StructField('storey_range', StringType(), True), StructField('floor_area_sqm', Dou

In [8]:
def load_dataframes(folder_path: str):
    folder = Path(folder_path)
    dfs = []
    for file in sorted(folder.glob("*.csv")):
        df = (
            spark.read
            .option("header", True)
            .option("inferSchema", True)
            .csv(str(file))
        )
        dfs.append((file.name, df))
    return dfs
dfs = load_dataframes("Raw")


In [9]:
schema_dict = {
    field.name: field.dataType
    for field in schema.fields
}
print(schema_dict)

{'month': TimestampType(), 'town': StringType(), 'flat_type': StringType(), 'block': StringType(), 'street_name': StringType(), 'storey_range': StringType(), 'floor_area_sqm': DoubleType(), 'flat_model': StringType(), 'lease_commence_date': IntegerType(), 'resale_price': DoubleType(), 'remaining_lease': StringType()}


In [10]:
def align_dataframes(dfs, schema):
    aligned_dfs = []
    for _, df in dfs:
        for field in schema.fields:
            if field.name not in df.columns:
                df = df.withColumn(
                    field.name,
                    lit(None).cast(field.dataType)
                )
        df = df.select([field.name for field in schema.fields])
        aligned_dfs.append(df)
    return aligned_dfs
aligned_dfs = align_dataframes(dfs, schema)


In [11]:
def build_master_dataframe(aligned_dataframes):
    return reduce(
        lambda a, b: a.unionByName(b),
        aligned_dataframes
    ).withColumn(
        "month",
        date_format(col("month"), "yyyy-MM")
    )


master_df = build_master_dataframe(aligned_dfs)


## 2. Data Profiling

This section profiles the combined dataset to understand quality issues before validation.

### Metrics checked
- Row and column counts
- Duplicate records
- Null counts and null percentages
- Distinct counts for categorical fields
- Summary statistics for numeric fields
- Date/month consistency checks

### Output
The profiling step produces a summary report to identify rows/columns that need cleaning before final transformation.


In [12]:
def profile_dataframe(df: DataFrame):
    print("=" * 80)
    print("DATA PROFILING REPORT")
    print("=" * 80)
    # ---------------------------------------------------------
    # 1. Dataset-level profile
    # ---------------------------------------------------------
    total_rows = df.count()
    total_columns = len(df.columns)
    print("\n[1] DATASET SUMMARY")
    print("-" * 80)
    print(f"Rows       : {total_rows:,}")
    print(f"Columns    : {total_columns:,}")
    print(f"Duplicates : {total_rows - df.dropDuplicates().count():,}")
    # ---------------------------------------------------------
    # 2. Schema
    # ---------------------------------------------------------
    print("\n[2] SCHEMA")
    print("-" * 80)
    df.printSchema()
    # ---------------------------------------------------------
    # 3. Column-level profiling
    # ---------------------------------------------------------
    print("\n[3] COLUMN PROFILE")
    print("-" * 80)
    profile = []
    for field in df.schema.fields:
        column = field.name
        null_count = df.filter(
            col(column).isNull()
        ).count()
        distinct_count = df.select(
            countDistinct(column)
        ).first()[0]
        null_percentage = (
            null_count / total_rows * 100
            if total_rows > 0
            else 0
        )
        profile.append((
            column,
            field.dataType.simpleString(),
            total_rows,
            null_count,
            round(null_percentage, 2),
            distinct_count
        ))
    profile_df = df.sparkSession.createDataFrame(
        profile,
        ["column",
            "data_type",
            "row_count",
            "null_count",
            "null_percentage",
            "distinct_count"])
    profile_df.show(total_columns,truncate=False)
    # ---------------------------------------------------------
    # 4. Numeric profiling
    # ---------------------------------------------------------
    numeric_columns = [
        field.name
        for field in df.schema.fields
        if field.dataType.simpleString()
        in ["int", "bigint", "double", "float", "long"]
    ]
    print("\n[4] NUMERIC PROFILE")
    print("-" * 80)
    for column in numeric_columns:
        print(f"\n{column}")
        df.select(
            count(column).alias("count"),
            min(column).alias("min"),
            max(column).alias("max"),
            avg(column).alias("mean"),
            stddev(column).alias("stddev")
        ).show(truncate=False)
        # Percentiles
        percentiles = df.approxQuantile(
            column,
            [0.01, 0.25, 0.50, 0.75, 0.99],
            0.01
        )
        print(
            f"Percentiles "
            f"[1%, 25%, 50%, 75%, 99%]: {percentiles}"
        )
    # ---------------------------------------------------------
    # 5. Categorical profiling
    # ---------------------------------------------------------
    categorical_columns = [
        field.name
        for field in df.schema.fields
        if field.dataType.simpleString() == "string"
    ]
    print("\n[5] CATEGORICAL PROFILE")
    print("-" * 80)
    for column in categorical_columns:
        distinct_count = df.select(
            countDistinct(column)
        ).first()[0]
        print(
            f"\n{column} "
            f"({distinct_count:,} distinct values)"
        )
        # Only display distributions for columns
        # with a reasonable number of categories
        if distinct_count <= 50:
            df.groupBy(column) \
                .count() \
                .orderBy(
                    col("count").desc()
                ) \
                .show(50, truncate=False)
    # ---------------------------------------------------------
    # 6. Resale-flat-specific data quality checks
    # ---------------------------------------------------------
    print("\n[6] DATA QUALITY CHECKS")
    print("-" * 80)
    # Resale price
    if "resale_price" in df.columns:
        invalid_price = df.filter(
            col("resale_price").isNull() |
            (col("resale_price") <= 0)
        ).count()
        print(f"Invalid resale prices : {invalid_price:,}")
    # Floor area
    if "floor_area_sqm" in df.columns:
        invalid_area = df.filter(
            col("floor_area_sqm").isNull() |
            (col("floor_area_sqm") <= 0)
        ).count()
        print(f"Invalid floor areas   : {invalid_area:,}")
    # Lease commencement year
    if "lease_commence_date" in df.columns:
        invalid_year = df.filter(
            col("lease_commence_date").isNull() |
            (col("lease_commence_date") < 1900) |
            (col("lease_commence_date") > 2026)
        ).count()
        print(f"Invalid lease years   : {invalid_year:,}")
    # Month format / date range
    if "month" in df.columns:
        invalid_month = df.filter(
            to_date(
                col("month"),
                "yyyy-MM"
            ).isNull()
        ).count()
        print(
            f"Invalid month values  : {invalid_month:,}"
        )
        date_range = df.select(
            min("month").alias("min_month"),
            max("month").alias("max_month")
        ).first()
        print(f"Minimum month         : {date_range['min_month']}")
        print(f"Maximum month         : {date_range['max_month']}")
        out_of_range = df.filter(
            (col("month") < "2012-01") |
            (col("month") > "2016-12")
        ).count()
        print(f"Outside required range of 2012 to 2016: {out_of_range:,}")
    print("\n" + "=" * 80)
    print("END OF PROFILING REPORT")
    print("=" * 80)
    # Return the column profile so it can be saved/exported
    return profile_df

In [13]:
profile_dataframe(master_df)

DATA PROFILING REPORT



[1] DATASET SUMMARY
--------------------------------------------------------------------------------
Rows       : 985,670
Columns    : 11


Duplicates : 1,929

[2] SCHEMA
--------------------------------------------------------------------------------
root
 |-- month: string (nullable = true)
 |-- town: string (nullable = true)
 |-- flat_type: string (nullable = true)
 |-- block: string (nullable = true)
 |-- street_name: string (nullable = true)
 |-- storey_range: string (nullable = true)
 |-- floor_area_sqm: double (nullable = true)
 |-- flat_model: string (nullable = true)
 |-- lease_commence_date: integer (nullable = true)
 |-- resale_price: double (nullable = true)
 |-- remaining_lease: string (nullable = true)


[3] COLUMN PROFILE
--------------------------------------------------------------------------------


+-------------------+---------+---------+----------+---------------+--------------+
|column             |data_type|row_count|null_count|null_percentage|distinct_count|
+-------------------+---------+---------+----------+---------------+--------------+
|month              |string   |985670   |0         |0.0            |440           |
|town               |string   |985670   |0         |0.0            |27            |
|flat_type          |string   |985670   |0         |0.0            |8             |
|block              |string   |985670   |0         |0.0            |2781          |
|street_name        |string   |985670   |0         |0.0            |597           |
|storey_range       |string   |985670   |0         |0.0            |25            |
|floor_area_sqm     |double   |985670   |0         |0.0            |234           |
|flat_model         |string   |985670   |0         |0.0            |34            |
|lease_commence_date|int      |985670   |0         |0.0            |58      

+------+----+-----+-----------------+------------------+
|count |min |max  |mean             |stddev            |
+------+----+-----+-----------------+------------------+
|985670|28.0|366.7|95.66376200959746|25.716482816731762|
+------+----+-----+-----------------+------------------+



...

Invalid lease years   : 0


Invalid month values  : 0


Minimum month         : 1990-01
Maximum month         : 2026-08


Outside required range of 2012 to 2016: 893,126

END OF PROFILING REPORT


DataFrame[column: string, data_type: string, row_count: bigint, null_count: bigint, null_percentage: double, distinct_count: bigint]

# Data Quality Requirements

## 3. Data Validation

This section validates the dataset against business rules.

### Validation rules applied
- Month format follows `YYYY-MM`
- Town, flat type, flat model, and storey range must be valid values from the reference month
- Records with invalid values are flagged and excluded later

### Purpose
Validation separates usable transactions from records that fail data-quality rules, improving the reliability of downstream analytics.


In [16]:
jan2012 = master_df.filter(col("month") == "2012-01")
valid_towns = jan2012.select("town").distinct()
valid_flat_types = jan2012.select("flat_type").distinct()
valid_flat_models = jan2012.select("flat_model").distinct()
valid_storey_ranges = jan2012.select("storey_range").distinct()

In [18]:
valid_town_list = [
    row["town"]
    for row in valid_towns.collect()
]

validated_df = master_df.withColumn(
    "valid_date",
    when(
        col("month").rlike(r"^\d{4}-(0[1-9]|1[0-2])$"),
        1
    ).otherwise(0)
).withColumn(
    "valid_town",
    when(col("town").isin(valid_town_list), 1).otherwise(0)
).withColumn(
    "valid_flat_type",
    when(col("flat_type").isin([row["flat_type"] for row in valid_flat_types.collect()]), 1).otherwise(0)
).withColumn(
    "valid_flat_model",
    when(col("flat_model").isin([row["flat_model"] for row in valid_flat_models.collect()]), 1).otherwise(0)
).withColumn(
    "valid_storey_range",
    when(col("storey_range").isin([row["storey_range"] for row in valid_storey_ranges.collect()]), 1).otherwise(0)
).withColumn(
    "valid_record",
    when(
        (col("valid_town") == 1) &
        (col("valid_flat_type") == 1) &
        (col("valid_flat_model") == 1) &
        (col("valid_storey_range") == 1) &
        (col("valid_date") == 1),
        1
    ).otherwise(0)
)
validated_df.filter(col("valid_record") == 1).show(10, truncate=False)

+-------+----------+---------+-----+----------------+------------+--------------+--------------+-------------------+------------+---------------+----------+----------+---------------+----------------+------------------+------------+
|month  |town      |flat_type|block|street_name     |storey_range|floor_area_sqm|flat_model    |lease_commence_date|resale_price|remaining_lease|valid_date|valid_town|valid_flat_type|valid_flat_model|valid_storey_range|valid_record|
+-------+----------+---------+-----+----------------+------------+--------------+--------------+-------------------+------------+---------------+----------+----------+---------------+----------------+------------------+------------+
|2000-01|ANG MO KIO|3 ROOM   |170  |ANG MO KIO AVE 4|07 TO 09    |69.0          |Improved      |1986               |147000.0    |NULL           |1         |1         |1              |1               |1                 |1           |
|2000-01|ANG MO KIO|3 ROOM   |174  |ANG MO KIO AVE 4|04 TO 06    |61

## 4. Compute Remaining Lease

This section calculates the remaining lease period for each resale transaction.

### Assumptions
- Lease commenced on 1 January of the year recorded in `lease_commence_date`
- Resale transaction month is treated as the first day of the month
- Lease tenure is assumed to be 99 years from commencement date
- `remaining_lease`  is calculated based on current date minus `lease_commence_date`
### Output
The workflow produces `remaining_lease` as a human-readable string such as `X years Y months` and removes intermediate date fields used for calculation.


In [ ]:
def compute_remaining_lease(df):
    return (
        df
        # Create the lease commencement date
        .withColumn(
            "lease_commence_full_date",
            make_date(
                col("lease_commence_date"),
                lit(1),
                lit(1)
            )
        )
        # Calculate the total number of months remaining
        # based on today's date
        .withColumn(
            "remaining_total_months",
            floor(
                months_between(
                    add_months(
                        col("lease_commence_full_date"),
                        99 * 12
                    ),
                    current_date()
                )
            )
        )
        # Format remaining lease as "XX years XX months"
        .withColumn(
            "remaining_lease",
            concat(
                floor(col("remaining_total_months") / 12),
                lit(" years "),
                (col("remaining_total_months") % 12),
                lit(" months")
            )
        )
    )
rf_validated_df = compute_remaining_lease(validated_df)

In [30]:
rf_validated_df.filter(col('month')=='2015-02').show(10, truncate=False)

+-------+----------+---------+-----+-----------------+------------+--------------+--------------+-------------------+------------+-----------------+----------+----------+---------------+----------------+------------------+------------+------------------------+----------------------+
|month  |town      |flat_type|block|street_name      |storey_range|floor_area_sqm|flat_model    |lease_commence_date|resale_price|remaining_lease  |valid_date|valid_town|valid_flat_type|valid_flat_model|valid_storey_range|valid_record|lease_commence_full_date|remaining_total_months|
+-------+----------+---------+-----+-----------------+------------+--------------+--------------+-------------------+------------+-----------------+----------+----------+---------------+----------------+------------------+------------+------------------------+----------------------+
|2015-02|ANG MO KIO|2 ROOM   |508  |ANG MO KIO AVE 8 |04 TO 06    |44.0          |Improved      |1980               |245000.0    |52 years 3 months|

## 5. Composite Key

This section creates a composite key to identify duplicate or repeated resale transactions.

### What it does
- Uses all relevant transaction attributes except the fields that are derived or validation-based.
- Excludes fields such as `resale_price`, `valid_record`, and related validation flags from the duplicate key.
- Applies a window function to rank records within each group using `resale_price` in descending order.
- Assigns `row_num` so duplicate records can be identified and filtered later.

### Why it matters
Duplicate records may appear across files or within the same source due to repeated transactions or data ingestion issues. A composite key allows the pipeline to group logically similar records together and retain the most representative row.

### Output
A `row_num` column is created for each group, which is later used to keep only the first valid record and discard repeated versions.


In [31]:
def deduplicate_records(df):
    composite_key = [
        column for column in df.columns
        if column not in {
            "resale_price",
            "valid_record",
            "valid_date",
            "valid_town",
            "valid_flat_type",
            "valid_flat_model",
            "valid_storey_range",
        }
    ]
    window_spec = (
        Window
        .partitionBy(composite_key)
        .orderBy(col("resale_price").desc())
    )
    return df.withColumn("row_num", row_number().over(window_spec))
deduplicated_df = deduplicate_records(rf_validated_df)


In [32]:
deduplicated_df.show(5)

+-------+----------+---------+-----+----------------+------------+--------------+--------------+-------------------+------------+-----------------+----------+----------+---------------+----------------+------------------+------------+------------------------+----------------------+-------+
|  month|      town|flat_type|block|     street_name|storey_range|floor_area_sqm|    flat_model|lease_commence_date|resale_price|  remaining_lease|valid_date|valid_town|valid_flat_type|valid_flat_model|valid_storey_range|valid_record|lease_commence_full_date|remaining_total_months|row_num|
+-------+----------+---------+-----+----------------+------------+--------------+--------------+-------------------+------------+-----------------+----------+----------+---------------+----------------+------------------+------------+------------------------+----------------------+-------+
|1990-01|ANG MO KIO|   3 ROOM|  128|ANG MO KIO AVE 3|    07 TO 09|          67.0|NEW GENERATION|               1978|     47000.

## 6. Identify Potentially anomalous resale price 

This section detects unusually high or low resale prices relative to similar transactions in the same market segment.

### Method
- Group records by `town`, `flat_type`, and `flat_model`.
- Compute the 25th percentile (`q1`), 75th percentile (`q3`), and interquartile range (`iqr`).
- Define the abnormal range using Tukey's rule:
  - `lower_bound = q1 - 1.5 * iqr`
  - `upper_bound = q3 + 1.5 * iqr`
- Flag any record where `resale_price` falls outside these bounds.
- Ignore groups that are too small to provide a reliable benchmark.

### Parameters
- `MIN_GROUP_SIZE = 10`
- `IQR_MULTIPLIER = 1.5`

### Why it matters
Resale prices can include outliers caused by data errors, premium transactions, or unusual property conditions. This method helps isolate suspicious records before final analysis.

### Output
A flag called `is_anomalous_price` is created:
- `1` = potentially anomalous
- `0` = not anomalous


In [34]:
def detect_anomalies(df):
    group_columns = ["town", "flat_type", "flat_model"]
    min_group_size = 10
    iqr_multiplier = 1.5
    price_stats = (
        df.groupBy(*group_columns)
        .agg(
            count("resale_price").alias("group_count"),
            percentile_approx("resale_price", 0.25).alias("q1"),
            percentile_approx("resale_price", 0.75).alias("q3")
        )
        .withColumn("iqr", col("q3") - col("q1"))
        .withColumn("lower_bound", col("q1") - iqr_multiplier * col("iqr"))
        .withColumn("upper_bound", col("q3") + iqr_multiplier * col("iqr"))
    )
    return (
        df.join(price_stats, on=group_columns, how="left")
        .withColumn(
            "is_anomalous_price",
            when(col("group_count") < min_group_size, 0)
            .when((col("resale_price") < col("lower_bound")) |
                  (col("resale_price") > col("upper_bound")), 1)
            .otherwise(0)
        )
    )
an_deduplicated_df = detect_anomalies(deduplicated_df)


In [35]:
an_deduplicated_df.show(5)

26/09/06 00:52:57 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+----------+---------+--------------+-------+-----+----------------+------------+--------------+-------------------+------------+-----------------+----------+----------+---------------+----------------+------------------+------------+------------------------+----------------------+-------+-----------+--------+--------+--------+-----------+-----------+------------------+
|      town|flat_type|    flat_model|  month|block|     street_name|storey_range|floor_area_sqm|lease_commence_date|resale_price|  remaining_lease|valid_date|valid_town|valid_flat_type|valid_flat_model|valid_storey_range|valid_record|lease_commence_full_date|remaining_total_months|row_num|group_count|      q1|      q3|     iqr|lower_bound|upper_bound|is_anomalous_price|
+----------+---------+--------------+-------+-----+----------------+------------+--------------+-------------------+------------+-----------------+----------+----------+---------------+----------------+------------------+------------+--------------------

## 7. Additional data validation rules
floor_area_sqm > 0
resale_price > 0
lease_commence_date is a valid 4-digit year

This section applies additional business-rule checks to ensure the dataset contains only valid and realistic resale transactions.

### Rules applied
- `floor_area_sqm > 0`
- `resale_price > 0`
- `lease_commence_date` falls within a realistic range, here defined as 1900 to 2100

### What is created
- `valid_floor_area`
- `valid_resale_price`
- `valid_lease_commence_date`
- `add_valid_record`

### Why it matters
These checks help filter out invalid records caused by data entry issues, missing values, or unrealistic transaction values. They reduce the chance of including corrupted or misleading rows in the final cleaned dataset.

### Output
A record is marked as valid only if all additional validation checks pass. This is later combined with earlier quality checks to produce the final cleaned result.


In [37]:
ad_dv_df = an_deduplicated_df.withColumn(
    "valid_floor_area",
    when(col("floor_area_sqm") > 0, 1).otherwise(0)
).withColumn(
    "valid_resale_price",
    when(col("resale_price") > 0, 1).otherwise(0)
).withColumn(
    "valid_lease_commence_date",
    when(
        col("lease_commence_date").between(1900, 2100),
        1
    ).otherwise(0)
).withColumn(
    "add_valid_record",
    when(
        (col("valid_floor_area") == 1) &
        (col("valid_resale_price") == 1) &
        (col("valid_lease_commence_date") == 1),
        1
    ).otherwise(0)
)

In [38]:
Cleaned = ad_dv_df.filter(
                    col('valid_record') == 1
                ).filter(
                    col('add_valid_record') == 1
                ).filter(
                    col('row_num') == 1            
                ).select(
                    'month', 
                    'town', 
                    'flat_type', 
                    'block',
                    'street_name',
                    'storey_range',
                    'floor_area_sqm', 
                    'flat_model', 
                    'lease_commence_date', 
                    'resale_price', 
                    'remaining_lease'
                )
Cleaned.show(10, truncate=False)

+-------+----------+---------+-----+-----------------+------------+--------------+--------------+-------------------+------------+-----------------+
|month  |town      |flat_type|block|street_name      |storey_range|floor_area_sqm|flat_model    |lease_commence_date|resale_price|remaining_lease  |
+-------+----------+---------+-----+-----------------+------------+--------------+--------------+-------------------+------------+-----------------+
|2000-01|ANG MO KIO|3 ROOM   |174  |ANG MO KIO AVE 4 |04 TO 06    |61.0          |Improved      |1986               |144000.0    |58 years 3 months|
|2000-01|ANG MO KIO|3 ROOM   |180  |ANG MO KIO AVE 5 |04 TO 06    |68.0          |New Generation|1981               |160100.0    |53 years 3 months|
|2000-01|ANG MO KIO|3 ROOM   |320  |ANG MO KIO AVE 1 |04 TO 06    |73.0          |New Generation|1977               |157000.0    |49 years 3 months|
|2000-01|ANG MO KIO|3 ROOM   |333  |ANG MO KIO AVE 1 |04 TO 06    |82.0          |New Generation|1981     

In [39]:
# Export the cleaned dataframe to CSV
# Spark writes a folder by default when using csv(); coalesce(1) keeps it to a single output part.
Cleaned.coalesce(1).write \
    .mode("overwrite") \
    .option("header", True) \
    .csv("output/Cleaned")
print("Cleaned dataset exported to output/Cleaned")

Cleaned dataset exported to output/Cleaned


# Data Transformation

This section prepares the cleaned dataset for downstream analysis and feature generation.

### Transformation steps
- Generate a standardised resale identifier
- Hash the identifier for a secure version

### Scope
The result is a transformed and hashed dataset suitable for analysis, reporting, and identifier-based tracking.


## 9. Create Resale Identifier Column

In [40]:
def build_resale_identifier(cleaned_df):
    avg_price = (
        cleaned_df
        .groupBy("month", "town", "flat_type")
        .agg(avg("resale_price").alias("avg_resale_price"))
    )
    av_ad_dv_df = cleaned_df.join(
        avg_price,
        on=["month", "town", "flat_type"],
        how="left"
    )
    return (
        av_ad_dv_df
        .withColumn("block_digits", regexp_replace(col("block"), "[^0-9]", ""))
        .withColumn(
            "block_3_digits",
            substring(lpad(col("block_digits"), 3, "0"), 1, 3)
        )
        .withColumn(
            "avg_price_2_digits",
            substring(floor(col("avg_resale_price")).cast("string"), 1, 2)
        )
        .withColumn("month_2_digits", substring(col("month"), 6, 2))
        .withColumn("town_initial", substring(col("town"), 1, 1))
        .withColumn(
            "Resale_Identifier",
            concat(
                lit("S"),
                col("block_3_digits"),
                col("avg_price_2_digits"),
                col("month_2_digits"),
                col("town_initial")
            )
        )
    )
hash_df = build_resale_identifier(Cleaned)


In [41]:
Transformed = hash_df.select(
                    'month', 
                    'town', 
                    'flat_type', 
                    'block',
                    'street_name',
                    'storey_range',
                    'floor_area_sqm', 
                    'flat_model', 
                    'lease_commence_date', 
                    'resale_price', 
                    'remaining_lease',
                    'Resale_Identifier'
                )
Transformed.show(10, truncate=False)

+-------+----------+---------+-----+-----------------+------------+--------------+--------------+-------------------+------------+-----------------+-----------------+
|month  |town      |flat_type|block|street_name      |storey_range|floor_area_sqm|flat_model    |lease_commence_date|resale_price|remaining_lease  |Resale_Identifier|
+-------+----------+---------+-----+-----------------+------------+--------------+--------------+-------------------+------------+-----------------+-----------------+
|2000-01|BEDOK     |3 ROOM   |102  |BEDOK NTH AVE 4  |04 TO 06    |82.0          |New Generation|1977               |235000.0    |49 years 3 months|S1021601B        |
|2000-01|BEDOK     |3 ROOM   |530  |BEDOK NTH ST 3   |04 TO 06    |68.0          |New Generation|1979               |155000.0    |51 years 3 months|S5301601B        |
|2000-01|BEDOK     |3 ROOM   |532  |BEDOK NTH ST 3   |10 TO 12    |68.0          |New Generation|1980               |170000.0    |52 years 3 months|S5321601B        

In [42]:
# Export the transformed dataframe to CSV
# Spark writes a folder by default when using csv(); coalesce(1) keeps it to a single output part.
Transformed.coalesce(1).write \
    .mode("overwrite") \
    .option("header", True) \
    .csv("output/Transformed")
print("Transformed dataset exported to output/Transformed")

Transformed dataset exported to output/Transformed


## 10. Hashed Dataset by creating Hash identifier using SHA-256

In [43]:
Hashed = Transformed.withColumn(
        "Hashed_Resale_Identifier",
        sha2(col("Resale_Identifier"), 256)
    )

In [44]:
Hashed.show(10, truncate=False)

+-------+----------+---------+-----+----------------+------------+--------------+--------------+-------------------+------------+-----------------+-----------------+----------------------------------------------------------------+
|month  |town      |flat_type|block|street_name     |storey_range|floor_area_sqm|flat_model    |lease_commence_date|resale_price|remaining_lease  |Resale_Identifier|Hashed_Resale_Identifier                                        |
+-------+----------+---------+-----+----------------+------------+--------------+--------------+-------------------+------------+-----------------+-----------------+----------------------------------------------------------------+
|2000-01|ANG MO KIO|EXECUTIVE|104A |ANG MO KIO ST 11|01 TO 03    |139.0         |Apartment     |1996               |527000.0    |68 years 3 months|S1045501A        |ae756f8a49b47375b8f5492a18a9f942629ed132d7b950cbaf9e894db43593cb|
|2000-01|ANG MO KIO|EXECUTIVE|104A |ANG MO KIO ST 11|10 TO 12    |146.0     

In [45]:
# Export the Hashed dataframe to CSV
# Spark writes a folder by default when using csv(); coalesce(1) keeps it to a single output part.
Hashed.coalesce(3).write \
    .mode("overwrite") \
    .option("header", True) \
    .csv("output/Hashed")
print("Hashed dataset exported to output/Hashed")

Hashed dataset exported to output/Hashed
